# 교시별 실습 Notebook

d3-p03-04

수업 페이지의 순서대로 실행합니다. 정답 블록과 선택 웹 자동화는 설명을 읽고 별도로 실행하세요.

In [ ]:
from pathlib import Path
import os

candidates = [Path.cwd(), *Path.cwd().parents]
course_root = next((p for p in candidates if (p / "raw/기존강사 수업자료").is_dir()), None)
if course_root is None:
    raise FileNotFoundError("수업 자료를 풀고 raw 폴더가 있는 위치에서 Notebook을 여세요.")
os.chdir(course_root)
print("수업 자료 폴더를 확인했습니다.")

## 선행 셀 준비

d3-p01-02의 기본 셀입니다. 이 Notebook을 단독으로 열어도 필요한 변수를 준비할 수 있도록 포함했습니다.

In [ ]:
from pathlib import Path
import pandas as pd

def load_prices(use_live=False):
    if use_live:
        import yfinance as yf
        result = yf.download("005930.KS", start="2025-01-01", end="2026-01-01",
                             auto_adjust=True, multi_level_index=False, progress=False)
        if result is None or result.empty:
            raise RuntimeError("주가 수집 결과가 비었습니다. 제공 CSV로 실습하세요.")
        result.index = pd.to_datetime(result.index).tz_localize(None)
    else:
        path = Path("raw/기존강사 수업자료/_ko_stock/stock_data/005930.csv")
        result = pd.read_csv(path, parse_dates=["Date"]).set_index("Date")
    result = result.sort_index()
    result = result.loc[~result.index.duplicated(keep="last")].copy()
    result["Close"] = pd.to_numeric(result["Close"], errors="raise")
    assert result.index.is_monotonic_increasing
    assert (result["Close"].dropna() > 0).all()
    return result

prices = load_prices(use_live=False)
Path("practice").mkdir(exist_ok=True)
print("행·열:", prices.shape)
print("기간:", prices.index.min().date(), "~", prices.index.max().date())
print(prices[["Close"]].head())

In [ ]:
frame = pd.DataFrame(index=prices.index)
frame["close"] = prices["Close"]
frame["return_1d"] = prices["Close"].pct_change(fill_method=None)
frame["ma_5"] = prices["Close"].rolling(5).mean()
frame["lag_close_1"] = prices["Close"].shift(1)
frame["target_next_close"] = prices["Close"].shift(-1)
frame["target_date"] = pd.Series(prices.index, index=prices.index).shift(-1)
frame = frame.dropna().copy()
print(frame.head().round(4))
print("학습 가능한 행:", len(frame))
feature_columns = ["close", "return_1d", "ma_5", "lag_close_1"]
X = frame[feature_columns]
y = frame["target_next_close"]
split = int(len(frame) * 0.8)
test_start = frame.index[split]
# 훈련 정답 날짜도 테스트 입력의 첫 날짜보다 앞서도록 경계 행을 제외합니다.
train_mask = (frame.index < test_start) & (frame["target_date"] < test_start)
test_mask = frame.index >= test_start
X_train, X_test = X.loc[train_mask], X.loc[test_mask]
y_train, y_test = y.loc[train_mask], y.loc[test_mask]
assert frame.loc[train_mask, "target_date"].max() < X_test.index.min()
print("훈련·테스트:", len(X_train), len(X_test))
print("훈련 마지막 정답 날짜:", frame.loc[train_mask, "target_date"].max().date())
print("테스트 첫 입력 날짜:", X_test.index.min().date())

In [ ]:
print(frame[["close", "lag_close_1", "target_next_close", "target_date"]].head(6))
print("종가 결측 수:", prices["Close"].isna().sum())
print("전체 / 훈련 / 경계 제외 / 테스트:", len(frame), len(X_train), len(frame) - len(X_train) - len(X_test), len(X_test))

# 요약

Linear Regression, Ridge, Lasso로 다음 거래일 종가를 예측하고 오늘 종가를 그대로 쓰는 기준과 MAE·RMSE·R²를 비교합니다.

## 3일차 3~4교시 — 회귀 모델과 기준 예측 비교

2026년 10월 8일(목) · 3일차 3~4교시 · 1차 초안. 공식 안내문은 일차별 단원과 총시간을 정하며, 이 페이지의 교시별 묶음은 수업 진행을 위한 제안입니다. 같은 실습을 이어가는 교시는 한 페이지로 묶었습니다.

## 시작 전에

1~2교시의 `day3_stock_regression` Notebook을 이어 씁니다. `pd`, `Path`, `frame`, `X_train`, `X_test`, `y_train`, `y_test`, `test_mask`가 필요합니다. Kernel 재시작 후에는 1~2교시의 모든 기본 셀을 먼저 실행합니다. pandas, scikit-learn, matplotlib을 사용합니다.

코드 블록 하나를 Notebook의 코드 셀 하나에 붙여 넣고 위에서 아래로 Shift+Enter로 실행합니다. 앞 셀을 실행하지 않고 뒤 셀부터 실행하면 변수를 찾지 못할 수 있습니다. 파일 경로는 수업 폴더 `kpc-lec`를 기준으로 합니다. 현재 위치가 다르면 파일 경로를 실제 파일 위치로 바꿉니다.

## 교시별 진행 제안

- 3교시: 회귀와 정규화 설명 → 훈련 자료로 표준화 → 세 모델 학습 → 오늘 종가 기준 예측.
- 4교시: 오차 지표 표 → 실제·예측 그림 → 기준 대비 개선 계산 → 프로젝트 제출물 정리 → 전체 과정 회고.

## 3. Linear Regression, Ridge, Lasso 모델 학습

Linear Regression은 선형 조합으로 종가를 예측합니다. Ridge와 Lasso는 계수가 커지는 것에 벌점을 더합니다. 입력 크기 차이를 줄이기 위해 훈련 자료 기준으로 표준화합니다. Lasso는 일부 계수를 0으로 만들 수 있습니다.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

regressors = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=10.0, max_iter=20000, tol=0.001),
}
regression_models = {}
predictions = {"Previous close baseline": X_test["close"].to_numpy()}
for name, estimator in regressors.items():
    model = Pipeline([("scale", StandardScaler()), ("model", estimator)])
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_test)
    regression_models[name] = model

테스트 자료에 `fit`하지 않습니다. alpha 값은 정규화 강도이며 두 모델의 같은 alpha가 같은 벌점 형태를 뜻하지는 않습니다. 이 예제의 값은 비교를 시작하기 위한 설정입니다. 정식 설정 선택은 훈련 기간 안의 시계열 검증으로 수행하고 마지막 테스트 기간은 최종 확인용으로 남깁니다.

## 4. 기준 모델 비교 및 성능 분석

주가 수준은 연속된 날에 비슷할 수 있으므로 **내일 종가도 오늘 종가와 같다**는 기준 예측과 비교합니다. 높은 R²만 보고 예측이 좋아졌다고 판단하지 않습니다.

In [ ]:
result_rows = []
for name, pred in predictions.items():
    result_rows.append({"model": name, "MAE": mean_absolute_error(y_test, pred),
                        "RMSE": root_mean_squared_error(y_test, pred), "R2": r2_score(y_test, pred)})
result_table = pd.DataFrame(result_rows).set_index("model")
print(result_table.round(3))
comparison = pd.DataFrame({"actual_next_close": y_test, **predictions}, index=X_test.index)
print(comparison.head().round(2))
comparison.to_csv("practice/stock_predictions.csv", index_label="input_date", encoding="utf-8-sig")

MAE는 절대 오차의 평균이고 RMSE는 큰 오차에 더 민감합니다. 둘 다 제공 CSV 종가와 같은 단위로 읽으며 작을수록 좋습니다. R²는 테스트 정답의 평균을 사용하는 통계적 기준과의 비교이고 음수가 될 수도 있습니다. 위의 전일 종가 기준 모델과는 다른 기준입니다.

**종합 실습:** 모델별 MAE를 기준 모델과 비교해 줄어든 금액을 계산합니다. 모델이 기준보다 나쁘면 “기준 모델을 개선하지 못했다”고 결과를 기록합니다. 산점도나 시계열 그림으로 오차가 큰 날짜를 찾아 질문을 만듭니다.

<details>
<summary>종합 실습 예시 코드</summary>

```python
import matplotlib.pyplot as plt
baseline_mae = result_table.loc["Previous close baseline", "MAE"]
print((baseline_mae - result_table["MAE"]).rename("MAE improvement").round(3))
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(frame.loc[test_mask, "target_date"], y_test, label="Actual next close")
ax.plot(frame.loc[test_mask, "target_date"], predictions["Ridge"], label="Ridge")
ax.set(title="Next trading day close (held-out period)", xlabel="Target date", ylabel="Price")
ax.legend()
fig.tight_layout()
Path("practice/plots").mkdir(parents=True, exist_ok=True)
fig.savefig("practice/plots/stock_regression.png", dpi=150)
plt.show()
plt.close(fig)
```

</details>

## 확인 문제

1. 오늘 종가를 입력과 정답에 동시에 넣으면 왜 안 되나요?
2. 시계열의 train/test를 무작위로 섞으면 어떤 문제가 생기나요?
3. R²가 높아도 전일 종가 기준 모델보다 오차가 크면 어떻게 보고하나요?

<details>
<summary>정답</summary>

1. 미래 예측이 아니라 같은 값을 복사하는 문제가 됩니다. 정답 시점을 다음 거래일로 명확히 정의해야 합니다.
2. 미래 자료로 훈련하고 과거를 테스트하는 상황이 섞여 실제 예측 시점의 평가가 되지 않습니다.
3. 주가 수준의 움직임은 설명해도 기준 예측의 오차를 개선하지 못했다고 기록합니다. R²와 MAE·RMSE를 함께 봅니다.

</details>

## 프로젝트 제출물

- 사용 파일 또는 수집 도구·기간·조정 옵션
- 예측 시점, 입력 열, 다음 거래일 Target 정의
- 날짜 순서와 훈련·테스트 경계 확인
- 기준 모델과 공식 세 모델의 MAE·RMSE·R² 표
- 오차 그림과 결과 해석 3문장

이 프로젝트는 시계열 회귀의 학습 절차를 익히는 실습입니다. 거래 비용이나 실제 투자 수익을 평가하는 실험은 포함하지 않습니다.

## 직접 해보기와 정답

가장 큰 오차 5개를 찾아 날짜와 실제값·예측값을 함께 확인합니다. 새로운 모델 튜닝에 테스트 기간을 반복 사용하지 않습니다.

In [ ]:
error_table = pd.DataFrame({
    "target_date": frame.loc[test_mask, "target_date"],
    "actual": y_test,
    "ridge_prediction": predictions["Ridge"],
}, index=X_test.index)
error_table["absolute_error"] = (error_table["actual"] - error_table["ridge_prediction"]).abs()
print(error_table.nlargest(5, "absolute_error").round(2))

마지막 보고 문장은 ① 예측 시점·분리 기준, ② 기준 모델 대비 MAE 변화, ③ 해석의 한계 순서로 씁니다.

<details>
<summary>결과 해석 예시</summary>

“오늘 장 마감 정보를 사용해 다음 거래일 종가를 예측했다. 과거 기간으로 훈련하고 뒤 80개 거래일 입력을 테스트했으며 경계의 미래 정답이 겹치는 한 행을 제외했다. Ridge의 MAE는 출력 표의 ○○이고 오늘 종가 기준의 ○○보다 ○○만큼 작았다/컸다. 이 결과는 해당 기간의 종가 오차 비교이며 거래 비용이나 실제 수익을 평가한 결과는 아니다.”

개선 값이 음수면 기준 예측보다 오차가 커졌다는 뜻입니다. 기준보다 나쁜 결과도 올바르게 보고하면 실습은 성공입니다.

</details>

## 흔한 오류와 해결

| 증상 | 해결 |
| --- | --- |
| root_mean_squared_error import 오류 | 아래 호환 코드로 RMSE를 계산하거나 scikit-learn 버전을 확인합니다. |
| Lasso 수렴 경고 | 입력 표준화가 적용됐는지 확인하고 max_iter 설정을 확인합니다. |
| NameError: X_train | 앞 페이지의 특성 생성·날짜 분리 셀부터 실행합니다. |
| R²가 음수 | 코드 오류로 단정하지 않습니다. 테스트 정답 평균을 쓰는 통계 기준보다 나쁠 수 있습니다. |
| 그림 날짜가 한 거래일 어긋남 | 그림의 x축은 입력 날짜가 아니라 target_date를 사용합니다. |

구버전 호환이 필요하면 해당 import 줄을 `from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score`로 바꾸고 RMSE 함수 호출을 `mean_squared_error(y_test, pred) ** 0.5`로 바꿉니다.

## 완료 체크

- [ ] 수업 폴더에서 uv로 Notebook을 열고 Kernel의 `sys.executable`이 수업 `.venv`인지 확인했습니다. [환경 안내](surl:14892)
- [ ] 실습을 저장한 뒤 Kernel을 재시작하고 선행 페이지의 준비 셀부터 위에서 아래로 다시 실행했습니다. 웹 수집은 저장한 자료와 성공·실패 기록을 구분했습니다.

- [ ] 기준 모델과 세 회귀 모델의 MAE·RMSE·R²를 같은 기간에서 계산했습니다.
- [ ] 기준보다 좋아졌는지 직접 숫자로 비교했습니다.
- [ ] 실제·예측 그림과 예측 CSV를 저장했습니다.
- [ ] 예측 시점·평가 결과·한계를 담은 세 문장을 썼습니다.
- [ ] Notebook을 저장하고 프로젝트 제출물 목록을 확인했습니다.

## 기준 자료

공식 안내문 3쪽의 「주가 데이터 기반 회귀 분석 프로젝트」 4시간 중 마지막 2교시를 기준으로 구성했습니다. 기존 단원 자료의 실습 코드를 교시별로 나눈 초안입니다. 실행 결과의 행 수는 제공 파일을 기준으로 하며, 라이브 수집 자료나 라이브러리 버전이 달라지면 값이 달라질 수 있습니다.

<!-- period-navigation:start -->
## 교시별 이동

[교시별 전체 안내](surl:14891) · [공식 단원별 안내](surl:14880) · [이전: 10월 8일 1~2교시 — 주가 수집과 특성 만들기](surl:14902)
<!-- period-navigation:end -->